# Unit 4 — Notebook 2: Agentic Workflows
## Introduction, Design Patterns, AutoGen & CrewAI

---

### What you will learn

| Section | Topic |
|---|---|
| 1 | What is Agentic AI? The ReAct loop explained |
| 2 | Agentic Design Patterns (Reflection, Planning, Tool Use, Multi-Agent) |
| 3 | AutoGen — Microsoft's conversational agent framework |
| 4 | CrewAI — Role-based multi-agent orchestration |
| 5 | Devyan — A full 4-agent software development pipeline |

### Why Agentic AI?

Standard LLM usage follows a **request–response** pattern:

```
User → [Single Prompt] → LLM → [Single Response] → Done
```

This works for simple tasks. But real-world tasks require:
- **Planning** — breaking a goal into steps
- **Tool use** — calling APIs, searching the web, writing files
- **Memory** — remembering what happened in previous steps
- **Error recovery** — retrying when a step fails
- **Collaboration** — multiple specialized agents working together

**Agentic AI** gives LLMs a loop: the ability to think → act → observe → think again.

```
AGENTIC LOOP (ReAct Pattern)
─────────────────────────────────────
         ┌──────────────────┐
         │    GOAL / TASK   │
         └────────┬─────────┘
                  │
         ┌────────▼─────────┐
    ┌───►│    THINK (LLM)   │  "What should I do next?"
    │    └────────┬─────────┘
    │             │
    │    ┌────────▼─────────┐
    │    │    ACT (Tool)    │  Search / Write / Call API
    │    └────────┬─────────┘
    │             │
    │    ┌────────▼─────────┐
    └────│  OBSERVE (Result)│  "What did I get back?"
         └────────┬─────────┘
                  │
         ┌────────▼─────────┐
         │  GOAL REACHED?   │  If yes → Output. If no → loop.
         └──────────────────┘
```

## Setup

In [ ]:
# pyautogen>=0.4 renamed the module to autogen_agentchat — pin to <0.4 for
# the classic ConversableAgent API used in this notebook.
# litellm is required by CrewAI to route to Groq (or any non-native provider).
# crewai checks for litellm at IMPORT TIME — if crewai was already imported
# before litellm was installed, you must restart the kernel for the check to pass.
!pip install -q "pyautogen<0.4" tavily-python crewai crewai-tools litellm python-dotenv

print("")
print("*** IMPORTANT: Restart the kernel now before running any other cell. ***")
print("    Kernel -> Restart Kernel  (or the restart button in the toolbar)")
print("    Then run all cells from the top.")
print("    Reason: crewai caches the litellm availability check at import time.")



*** IMPORTANT: Restart the kernel now before running any other cell. ***
    Kernel -> Restart Kernel  (or the restart button in the toolbar)
    Then run all cells from the top.
    Reason: crewai caches the litellm availability check at import time.


In [ ]:
import os
from dotenv import load_dotenv
from google.colab import userdata

load_dotenv()  # loads from .env file if present

# Set your API keys here if not using a .env file
#os.environ["GROQ_API_KEY"] = "your_groq_key_here"
#os.environ["TAVILY_API_KEY"] = "your_tavily_key_here"

#GROQ_API_KEY   = os.getenv("GROQ_API_KEY")
#TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
TAVILY_API_KEY = userdata.get("TAVILY_API_KEY")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

print(f"GROQ_API_KEY:   {'set' if GROQ_API_KEY else 'NOT SET — add to .env'}")
print(f"TAVILY_API_KEY: {'set' if TAVILY_API_KEY else 'NOT SET — add to .env'}")

GROQ_API_KEY:   set
TAVILY_API_KEY: set


---

## Part 1: Agentic Design Patterns

Before writing code, let's understand the four core design patterns that underpin all agentic frameworks.

### Pattern 1: Reflection

The agent critiques its own output and iteratively improves it.

```
Agent writes code
    ↓
Agent reviews its own code → finds bugs
    ↓
Agent rewrites improved code
    ↓
Agent reviews again → passes → done
```

### Pattern 2: Tool Use

The agent has access to external functions (tools) it can call during reasoning.

```
Thought: "I need to find the current stock price of Apple"
Action: search_web("Apple stock price today")
Observation: "AAPL: $189.30 as of market close"
Thought: "Now I can answer the user's question."
Final Answer: "Apple's stock is $189.30"
```

### Pattern 3: Planning

The agent breaks a complex goal into a sequence of sub-tasks before executing any of them.

```
Goal: "Write a research report on climate change"

Plan:
  1. Search for recent statistics on global temperature rise
  2. Search for key causes of climate change
  3. Search for proposed solutions
  4. Synthesize into a structured report
  5. Add citations

→ Execute each step in order
```

### Pattern 4: Multi-Agent Collaboration

Multiple specialized agents each handle what they're best at, coordinated by an orchestrator.

```
                  ┌─────────────────┐
                  │   ORCHESTRATOR  │
                  └────────┬────────┘
              ┌────────────┼────────────┐
              ▼            ▼            ▼
        [Researcher]  [Writer]    [Reviewer]
        finds facts   drafts      critiques
                           ↓
                      Final output
```

All the frameworks we cover below implement these patterns — just with different APIs and abstractions.

---

## Part 2: AutoGen — Conversational Multi-Agent Framework

### What is AutoGen?

**AutoGen** (by Microsoft) is a framework where agents communicate through a **conversation protocol**. Each agent is a participant in a chat — they send and receive messages, and can call tools (functions) when needed.

```
AutoGen Core Concepts:

  ConversableAgent  ← the base agent class
       ├─ AssistantAgent  (has an LLM, generates responses)
       └─ UserProxyAgent  (represents the human, executes tools)

  register_function()  ← attach a Python function as a callable tool
  initiate_chat()      ← start the conversation between two agents
```

### Our Use Case: Research → Write → Translate Pipeline

We'll build a 3-agent pipeline that:
1. **Researcher** searches the web for a given topic
2. **Writer** turns the findings into a structured article
3. **Translator** translates the article into another language

Each step is a separate `initiate_chat()` call, passing outputs from one agent to the next.

In [ ]:
from autogen import ConversableAgent
from tavily import TavilyClient

# ── LLM Configuration ──────────────────────────────────────────────────────
# Note: AutoGen 0.3.x sends tools using the old OpenAI 'functions' schema,
# which Groq rejects with a 400 error regardless of model.
# Solution: we handle the web search in plain Python (Tavily), inject the
# results into the researcher's message, then let AutoGen handle the
# multi-agent conversation (synthesise → write → translate).
# This is cleaner and teaches the same multi-agent concepts.
groq_config = [
    {
        "model": "llama-3.3-70b-versatile",
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",
        "api_type": "openai",
    }
]

llm_config = {"config_list": groq_config, "temperature": 0.3}
print("LLM config ready — using Groq / llama-3.3-70b-versatile")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


LLM config ready — using Groq / llama-3.3-70b-versatile


In [ ]:
# ── Tool: Web Search via Tavily ─────────────────────────────────────────────
def search_internet(query: str) -> str:
    """
    Search the web using the Tavily API.

    Args:
        query (str): The search query string.

    Returns:
        str: Search results as a formatted string.
    """
    client = TavilyClient(api_key=TAVILY_API_KEY)
    try:
        response = client.search(query=query, max_results=5)
        # Format results cleanly
        results = []
        for r in response.get("results", []):
            results.append(f"Title: {r.get('title', '')}\nURL: {r.get('url', '')}\nContent: {r.get('content', '')}")
        return "\n\n---\n\n".join(results)
    except Exception as e:
        return f"Search error: {str(e)}"

print("search_internet tool defined.")

search_internet tool defined.


In [ ]:
# ── Plain Python web search (Tavily) ────────────────────────────────────────
# We call Tavily directly in Python instead of through AutoGen's tool system.
# AutoGen 0.3.x uses the deprecated 'functions' API which Groq no longer accepts.
def run_web_search(query: str, max_results: int = 5) -> str:
    """Run a Tavily web search and return formatted results."""
    client = TavilyClient(api_key=TAVILY_API_KEY)
    try:
        response = client.search(query=query, max_results=max_results)
        results = []
        for r in response.get("results", []):
            results.append(
                f"Title: {r.get('title', '')}\n"
                f"URL: {r.get('url', '')}\n"
                f"Content: {r.get('content', '')}"
            )
        return "\n\n---\n\n".join(results)
    except Exception as e:
        return f"Search error: {str(e)}"

print("Web search function ready.")

# ── Agent 1: User Proxy ─────────────────────────────────────────────────────
# The UserProxy acts as the human — no LLM, terminates on 'TERMINATE'.
user_proxy = ConversableAgent(
    name="UserProxy",
    llm_config=False,
    human_input_mode="NEVER",
    code_execution_config=False,
    is_termination_msg=lambda x: "TERMINATE" in (x.get("content") or ""),
    default_auto_reply="Continue if not finished, otherwise say TERMINATE."
)

# ── Agent 2: Researcher ─────────────────────────────────────────────────────
# Receives pre-fetched search results and synthesises them into a report.
researcher_agent = ConversableAgent(
    name="Researcher",
    llm_config=llm_config,
    system_message=(
        "You are a professional research analyst. "
        "You will be given raw web search results on a topic. "
        "Synthesise them into a structured research report with: key facts, "
        "important statistics, key developments, and source URLs. "
        "When your report is complete, end your final message with 'TERMINATE'."
    )
)

# ── Agent 3: Writer ─────────────────────────────────────────────────────────
writer_agent = ConversableAgent(
    name="Writer",
    llm_config=llm_config,
    system_message=(
        "You are a professional content writer. "
        "Take research findings and transform them into a well-structured, "
        "engaging article with: an introduction, 3-4 clearly titled sections, "
        "key takeaways, and proper source citations. "
        "When your article is complete, end with 'TERMINATE'."
    )
)

# ── Agent 4: Translator ─────────────────────────────────────────────────────
translator_agent = ConversableAgent(
    name="Translator",
    llm_config=llm_config,
    system_message=(
        "You are a professional translator. "
        "Translate the given article into Spanish, preserving the original "
        "meaning, tone, structure, and all section headings. "
        "When the translation is complete, end with 'TERMINATE'."
    )
)

print("All agents created.")

Web search function ready.
All agents created.


In [ ]:
# ── Step 1: Search (Python) then Research (AutoGen) ────────────────────────
RESEARCH_TOPIC = "Recent advances in large language models in 2024 and 2025"

# Step 1a — Run the actual web search in plain Python
print(f"Searching the web for: '{RESEARCH_TOPIC}'...")
raw_search_results = run_web_search(RESEARCH_TOPIC, max_results=5)
print(f"Search complete. Got {len(raw_search_results)} characters of results.")

# Step 1b — Hand results to the Researcher agent to synthesise into a report
# The agent receives the raw results and uses its LLM to analyse + structure them.
print(f"\nResearcher agent synthesising report...")
print("=" * 60)

research_result = user_proxy.initiate_chat(
    researcher_agent,
    message=(
        f"Topic: {RESEARCH_TOPIC}\n\n"
        f"Here are the raw web search results:\n\n{raw_search_results}\n\n"
        "Please synthesise these into a structured research report with key facts, "
        "important statistics, and source URLs."
    ),
    max_turns=4
)

research_findings = research_result.chat_history[-1]["content"].replace("TERMINATE", "").strip()
print("\nResearch complete.")

Searching the web for: 'Recent advances in large language models in 2024 and 2025'...
Search complete. Got 1562 characters of results.

Researcher agent synthesising report...
UserProxy (to Researcher):

Topic: Recent advances in large language models in 2024 and 2025

Here are the raw web search results:

Title: Large Language Models That Will Redefine AI in 2025 - GrowExx
URL: https://www.growexx.com/blog/large-language-models-that-will-redefine-ai-in-2025/
Content: Explore how large language models will revolutionize AI in 2025, advancing automation, creativity, and transforming business applications.

---

Title: Top Large Language Models of 2025 | Best LLMs Compared - Nurix AI
URL: https://www.nurix.ai/blogs/which-llm-most-advanced-today
Content: In 2025, large language models (LLMs) are powering everything from AI agents to real-time analytics. With rapid advancements in reasoning,

---

Title: Advances in Large Language Model Training: Insights from ICLR ...
URL: https://www.pap

Researcher (to UserProxy):

**Research Report: Recent Advances in Large Language Models in 2024 and 2025**

**Key Facts:**

1. Large language models (LLMs) are expected to revolutionize AI in 2025, advancing automation, creativity, and transforming business applications.
2. LLMs are powering various applications, including AI agents and real-time analytics, with rapid advancements in reasoning.
3. Recent publications at ICLR 2025 have significantly advanced our understanding of LLM training.
4. LLMs are being used to identify patterns from complex biological data, which is a trend to watch in 2025.
5. The state of LLMs in 2025 includes progress, problems, and predictions, with a focus on inference-time scaling, benchmarks, architectures, and new models such as DeepSeek R1 and RLVR.

**Important Statistics:**

* No specific statistics are mentioned in the provided search results. However, the reports and reviews mentioned in the search results may contain relevant statistics and data.



In [ ]:
# ── Step 2: Write ───────────────────────────────────────────────────────────
print("Starting article writing...")
print("=" * 60)

writing_result = user_proxy.initiate_chat(
    writer_agent,
    message=f"Based on the following research findings, write a well-structured article:\n\n{research_findings}",
    max_turns=4
)

article = writing_result.chat_history[-1]["content"].replace("TERMINATE", "").strip()
print("\nArticle writing complete.")

Starting article writing...
UserProxy (to Writer):

Based on the following research findings, write a well-structured article:

**Research Report: Recent Advances in Large Language Models in 2024 and 2025**

**Key Facts:**

1. Large language models (LLMs) are expected to revolutionize AI in 2025, advancing automation, creativity, and transforming business applications.
2. LLMs are powering various applications, including AI agents and real-time analytics, with rapid advancements in reasoning.
3. Recent publications at ICLR 2025 have significantly advanced our understanding of LLM training.
4. LLMs are being used to identify patterns from complex biological data, which is a trend to watch in 2025.
5. The state of LLMs in 2025 includes progress, problems, and predictions, with a focus on inference-time scaling, benchmarks, architectures, and new models such as DeepSeek R1 and RLVR.

**Important Statistics:**

* No specific statistics are mentioned in the provided search results. However,

Writer (to UserProxy):

**Introduction to Large Language Models: Revolutionizing AI in 2025**

The field of artificial intelligence (AI) is on the cusp of a revolution, driven by the rapid advancements in large language models (LLMs). These models are expected to transform the way we approach automation, creativity, and business applications in 2025. With their ability to power various applications, including AI agents and real-time analytics, LLMs are poised to make a significant impact on various industries. Recent publications at the International Conference on Learning Representations (ICLR) 2025 have significantly advanced our understanding of LLM training, paving the way for improved performance and efficiency.

**Advances in LLM Training and Models**

The ICLR 2025 publications have highlighted significant advancements in LLM training, which is expected to improve the performance and efficiency of LLMs. New models such as DeepSeek R1 and RLVR are being developed, which may offer

In [ ]:
# ── Step 3: Translate ───────────────────────────────────────────────────────
print("Starting translation to Spanish...")
print("=" * 60)

translation_result = user_proxy.initiate_chat(
    translator_agent,
    message=f"Please translate the following article into Spanish:\n\n{article}",
    max_turns=4
)

translated_article = translation_result.chat_history[-1]["content"].replace("TERMINATE", "").strip()
print("\nTranslation complete.")

Starting translation to Spanish...
UserProxy (to Translator):

Please translate the following article into Spanish:

**Introduction to Large Language Models: Revolutionizing AI in 2025**

The field of artificial intelligence (AI) is on the cusp of a revolution, driven by the rapid advancements in large language models (LLMs). These models are expected to transform the way we approach automation, creativity, and business applications in 2025. With their ability to power various applications, including AI agents and real-time analytics, LLMs are poised to make a significant impact on various industries. Recent publications at the International Conference on Learning Representations (ICLR) 2025 have significantly advanced our understanding of LLM training, paving the way for improved performance and efficiency.

**Advances in LLM Training and Models**

The ICLR 2025 publications have highlighted significant advancements in LLM training, which is expected to improve the performance and eff

Translator (to UserProxy):

**Introducción a los Grandes Modelos de Lenguaje: Revolucionando la IA en 2025**

El campo de la inteligencia artificial (IA) está al borde de una revolución, impulsada por los rápidos avances en los grandes modelos de lenguaje (GML). Estos modelos se espera que transformen la forma en que abordamos la automatización, la creatividad y las aplicaciones comerciales en 2025. Con su capacidad para impulsar diversas aplicaciones, incluyendo agentes de IA y análisis en tiempo real, los GML están en posición de tener un impacto significativo en diversas industrias. Las publicaciones recientes en la Conferencia Internacional sobre Representaciones de Aprendizaje (ICLR) 2025 han avanzado significativamente nuestra comprensión de la formación de GML, allanando el camino para un mejor rendimiento y eficiencia.

**Avances en la Formación y Modelos de GML**

Las publicaciones de la ICLR 2025 han destacado avances significativos en la formación de GML, lo que se espera qu

In [ ]:
# ── Final Output ────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("PIPELINE COMPLETE — Final Outputs")
print("=" * 60)

print("\n--- RESEARCH FINDINGS (excerpt) ---")
print(research_findings[:800] + "..." if len(research_findings) > 800 else research_findings)

print("\n--- ARTICLE (excerpt) ---")
print(article[:800] + "..." if len(article) > 800 else article)

print("\n--- TRANSLATED ARTICLE (excerpt) ---")
print(translated_article[:800] + "..." if len(translated_article) > 800 else translated_article)


PIPELINE COMPLETE — Final Outputs

--- RESEARCH FINDINGS (excerpt) ---
**Research Report: Recent Advances in Large Language Models in 2024 and 2025**

**Key Facts:**

1. Large language models (LLMs) are expected to revolutionize AI in 2025, advancing automation, creativity, and transforming business applications.
2. LLMs are powering various applications, including AI agents and real-time analytics, with rapid advancements in reasoning.
3. Recent publications at ICLR 2025 have significantly advanced our understanding of LLM training.
4. LLMs are being used to identify patterns from complex biological data, which is a trend to watch in 2025.
5. The state of LLMs in 2025 includes progress, problems, and predictions, with a focus on inference-time scaling, benchmarks, architectures, and new models such as DeepSeek R1 and RLVR.

**Important Statistics:**

* No sp...

--- ARTICLE (excerpt) ---
**Introduction to Large Language Models: Revolutionizing AI in 2025**

The field of artificial in

### What just happened?

```
AutoGen Pipeline — What happened under the hood:

  [Python]  Tavily.search("LLM advances 2024") ──► Raw search results
                                                           │
                                           injected into first message
                                                           │
  UserProxy ──► Researcher: "Here are raw results, synthesise a report"
  Researcher ──► [LLM analyses and structures the data]
  Researcher ──► "Here are the findings... TERMINATE"

  UserProxy ──► Writer: "Write an article from these findings"
  Writer ──► "Here is the article... TERMINATE"

  UserProxy ──► Translator: "Translate to Spanish"
  Translator ──► "Aquí está el artículo... TERMINATE"
```

> **Note on tool calling:** AutoGen 0.3.x uses the deprecated OpenAI `functions` schema
> for tool registration, which Groq no longer accepts. Rather than fight framework
> internals, we call Tavily directly in Python and inject the results — this is actually
> the pattern used in many production pipelines where tool execution is handled outside
> the LLM loop for reliability and auditability.

Each agent had a **single responsibility**, communicated via messages, and knew when to stop (`TERMINATE`). The `UserProxy` served as the glue — passing outputs from one agent as the input to the next.

---

## Part 3: CrewAI — Role-Based Agent Orchestration

### What is CrewAI?

**CrewAI** models agent collaboration as a **crew** — like a team of people at work. Each crew member has:
- A **role** (job title)
- A **goal** (what they're trying to achieve)
- A **backstory** (personality/expertise context for the LLM)
- **Tools** (what they can use)
- **Tasks** (the specific work assigned to them)

```
CrewAI Concepts:

  Agent(role, goal, backstory, tools, llm)
      ↑ the "who"

  Task(description, agent, expected_output, context)
      ↑ the "what"

  Crew(agents, tasks, process)
      ↑ the "how" — sequential or hierarchical

  crew.kickoff() → runs the whole pipeline
```

### AutoGen vs CrewAI

| | AutoGen | CrewAI |
|---|---|---|
| Mental model | Agents in a conversation | Crew members on a project |
| Coordination | Chat messages | Tasks with explicit dependencies |
| Control flow | `initiate_chat()` | `crew.kickoff()` |
| Best for | Back-and-forth reasoning | Structured pipelines with clear roles |
| Tool definition | `register_function()` | `@tool` decorator |

In [ ]:
import os
from crewai import Agent, Task, Crew, LLM
from crewai.tools import tool
from tavily import TavilyClient
import time

# litellm reads GROQ_API_KEY from environment — set it explicitly here
# so it works even if the .env file was not loaded before crewai imported.
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# llama-3.1-8b-instant: 560 t/s, ~1M TPM on free tier (vs 12K for 70b).
# max_tokens=800 caps per-response size to stay well under rate limits.
crew_llm = LLM(
    model="groq/llama-3.1-8b-instant",
    temperature=0.3,
    max_tokens=800,
    api_key=GROQ_API_KEY
)

print("CrewAI LLM configured — groq/llama-3.1-8b-instant")


CrewAI LLM configured — groq/llama-3.1-8b-instant


In [ ]:
# ── Tool: Web Search (CrewAI style with @tool decorator) ────────────────────
@tool("Web Search Tool")
def crew_search_internet(query: str) -> str:
    """
    Search the web for the given query using the Tavily API.
    Always pass a clear, specific search query string as input.

    Args:
        query (str): The search query.

    Returns:
        str: Web search results including titles, URLs, and content snippets.
    """
    client = TavilyClient(api_key=TAVILY_API_KEY)
    try:
        response = client.search(query=query, max_results=5)
        results = []
        for r in response.get("results", []):
            results.append(f"Title: {r.get('title', '')}\nURL: {r.get('url', '')}\nContent: {r.get('content', '')}")
        return "\n\n---\n\n".join(results)
    except Exception as e:
        return f"Search error: {str(e)}"

print("CrewAI search tool defined.")

CrewAI search tool defined.


In [ ]:
# Agents with short backstories to minimise prompt token usage.
researcher = Agent(
    role="Research Analyst",
    goal="Find key facts using web search and return exactly 5 bullet points with source URLs.",
    backstory="Expert researcher who produces brief bullet-point summaries.",
    tools=[crew_search_internet],
    verbose=False,
    llm=crew_llm
)
writer = Agent(
    role="Content Writer",
    goal="Turn bullet points into a 150-word article with 4 paragraphs.",
    backstory="Concise tech writer who produces short, structured articles.",
    verbose=False,
    llm=crew_llm
)
translator = Agent(
    role="Translator",
    goal="Translate the given English text into Spanish accurately.",
    backstory="Professional English-Spanish translator.",
    verbose=False,
    llm=crew_llm
)
print("Agents defined.")


Agents defined.


In [ ]:
TOPIC = "Recent advances in large language models in 2024 and 2025"

research_task = Task(
    description=(
        f"Search for: {TOPIC}. "
        "Return ONLY 5 bullet points. Each bullet: one key fact + source URL. "
        "Each bullet must be under 25 words. No intro, no conclusion."
    ),
    agent=researcher,
    expected_output="5 bullet points, each under 25 words, with source URL."
)
writing_task = Task(
    description=(
        "Using the 5 research bullets, write a SHORT article: "
        "intro (1 sentence), 2 body paragraphs (2 sentences each), conclusion (1 sentence). "
        "Total: 80-100 words maximum."
    ),
    agent=writer,
    context=[research_task],
    expected_output="An 80-100 word article, 4 paragraphs."
)
translation_task = Task(
    description="Translate the article into Spanish. Output the translation only.",
    agent=translator,
    context=[writing_task],
    expected_output="The article translated into Spanish."
)
print("Tasks defined.")


Tasks defined.


In [ ]:
import time

def run_with_retry(crew_obj, max_attempts=5):
    for attempt in range(1, max_attempts + 1):
        try:
            return crew_obj.kickoff()
        except Exception as e:
            err = str(e)
            if "rate_limit" in err.lower() or "429" in err or "RateLimitError" in err:
                wait = attempt * 30
                print(f"  Rate limit hit (attempt {attempt}/{max_attempts}). Sleeping {wait}s...")
                time.sleep(wait)
            else:
                raise
    print("All retries exhausted. Wait a minute then re-run this cell.")
    return None

crew = Crew(
    agents=[researcher, writer, translator],
    tasks=[research_task, writing_task, translation_task],
    verbose=False
)
print("Starting crew...")
crew_result = run_with_retry(crew)
if crew_result:
    print("Done.")


Starting crew...


[CrewAIEventsBus] Warning: Event pairing mismatch. 'agent_execution_completed' closed 'llm_call_started' (expected 
'agent_execution_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_completed' closed 'agent_execution_started' (expected 
'task_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')

Done.




In [ ]:
if crew_result is None:
    print("No result — rate limit retries exhausted. Re-run the kickoff cell.")
else:
    print("" + "=" * 60)
    print("CREWAI RESULTS")
    print("=" * 60)
    print("--- RESEARCH REPORT ---")
    print(str(crew_result.tasks_output[0])[:800])
    print("--- ARTICLE ---")
    print(str(crew_result.tasks_output[1])[:800])
    print("--- SPANISH TRANSLATION ---")
    print(str(crew_result.tasks_output[2])[:800])


CREWAI RESULTS
--- RESEARCH REPORT ---
The requirements are not met as the previous response contained more than 5 bullet points and some of the bullet points exceeded 25 words.


--- ARTICLE ---
Here's a rewritten article based on the 5 research bullets provided.

The increasing demand for cybersecurity measures has led to the development of advanced threat detection systems.

These systems utilize machine learning algorithms to identify patterns in network traffic, enabling them to detect potential threats more effectively. This approach has proven to be more accurate than traditional signature-based detection methods.

The use of artificial intelligence in cybersecurity has also led to the development of predictive analytics tools. These tools can forecast potential security breaches, allowing organizations to take proactive measures to prevent them.

As a result, the integration of AI and machine learning in cybersecurity has become a crucial aspect of modern threat detection.
--- 

### What just happened?

```
CrewAI Pipeline — Under the hood:

  crew.kickoff()
       │
       ├─► Task 1 → researcher runs → uses search tool → produces report
       │         └─ output fed as context into Task 2
       │
       ├─► Task 2 → writer runs → reads research context → writes article
       │         └─ output fed as context into Task 3
       │
       └─► Task 3 → translator runs → reads article context → translates

  CrewAI manages the dependency chain automatically via context=[...]
```

Notice how CrewAI's `context=[task]` is more explicit than AutoGen's sequential `initiate_chat()` calls — the dependencies are declared upfront rather than managed manually.

---

## Part 4: Devyan — A 4-Agent Software Development Pipeline

### What is Devyan?

**Devyan** is a multi-agent software development assistant. Instead of a single agent writing code (which leads to bugs, poor architecture, and no testing), Devyan uses **four specialized agents** that mirror a real software team:

```
DEVYAN PIPELINE
──────────────────────────────────────────────────────

  User: "Build a Python function to sort a list of dicts by key"
          │
          ▼
  ┌─────────────────┐
  │  ARCHITECT      │  Designs the solution architecture
  │  Agent          │  "We need: input validation, sort function,
  └────────┬────────┘   edge case handling..."
           │ architecture doc
           ▼
  ┌─────────────────┐
  │  PROGRAMMER     │  Implements the code per architecture
  │  Agent          │  Writes working Python code
  └────────┬────────┘
           │ code
           ▼
  ┌─────────────────┐
  │  TESTER         │  Writes and runs test cases
  │  Agent          │  Unit tests, edge cases, assertions
  └────────┬────────┘
           │ test results
           ▼
  ┌─────────────────┐
  │  REVIEWER       │  Reviews everything and produces final docs
  │  Agent          │  Code quality, test coverage, usage instructions
  └─────────────────┘
           │
           ▼
  Final: Code + Tests + Documentation
──────────────────────────────────────────────────────
```

In [ ]:
import os
from crewai import Agent, Task, Crew, LLM
from textwrap import dedent
import time

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# llama-3.1-8b-instant: faster and higher rate limits than 70b.
# max_tokens=800 caps code output size to avoid TPM exhaustion.
devyan_llm = LLM(
    model="groq/llama-3.1-8b-instant",
    temperature=0.2,
    max_tokens=800,
    api_key=GROQ_API_KEY
)

print("Devyan LLM configured — groq/llama-3.1-8b-instant")


Devyan LLM configured — groq/llama-3.1-8b-instant


In [ ]:
# Short backstories to minimise token usage.
architect = Agent(
    role="Software Architect",
    goal="Output a function signature, parameter descriptions, return type, and 3 edge cases. Max 100 words.",
    backstory="Architect who writes concise design notes.",
    allow_delegation=False, verbose=False, llm=devyan_llm
)
programmer = Agent(
    role="Python Developer",
    goal="Implement the function as a single clean Python code block with type hints and docstring.",
    backstory="Python developer who writes concise, working code.",
    allow_delegation=False, verbose=False, llm=devyan_llm
)
tester = Agent(
    role="QA Engineer",
    goal="Write exactly 5 pytest test functions as a single code block. No explanations.",
    backstory="QA engineer who writes concise targeted tests.",
    allow_delegation=False, verbose=False, llm=devyan_llm
)
reviewer = Agent(
    role="Code Reviewer",
    goal="Output 3 feedback bullets and one short usage paragraph. Max 80 words total.",
    backstory="Tech lead who writes brief, actionable reviews.",
    allow_delegation=False, verbose=False, llm=devyan_llm
)
print("Devyan agents defined.")


Devyan agents defined.


In [ ]:
from crewai import Task
USER_PROBLEM = (
    "Build process_students(records: list[dict], min_score: float) -> dict that "
    "filters by min_score, sorts descending, returns {filtered, average, top_student}."
)

architecture_task = Task(
    description=(
        f"Design: {USER_PROBLEM} "
        "Output: function signature with type hints, param descriptions, return type, "
        "3 edge cases. Maximum 100 words."
    ),
    agent=architect,
    expected_output="Function signature, params, return, 3 edge cases. Under 100 words."
)
implementation_task = Task(
    description=(
        "Implement process_students() per the architecture. "
        "Output ONLY a Python code block. Type hints, docstring, edge cases. No prose."
    ),
    agent=programmer,
    context=[architecture_task],
    expected_output="A Python code block only."
)
testing_task = Task(
    description=(
        "Write exactly 5 pytest test functions for process_students(). "
        "Output ONLY the test code block. No prose, no explanations."
    ),
    agent=tester,
    context=[implementation_task],
    expected_output="Exactly 5 pytest functions as a code block."
)
reviewing_task = Task(
    description=(
        "Review the code and tests. Output ONLY: "
        "3 bullet points of feedback, then 1 sentence on how to run it. "
        "Max 80 words total."
    ),
    agent=reviewer,
    context=[architecture_task, implementation_task, testing_task],
    expected_output="3 bullets + 1 sentence. Under 80 words."
)
print("Devyan tasks defined.")


Devyan tasks defined.


In [ ]:
devyan_crew = Crew(
    agents=[architect, programmer, tester, reviewer],
    tasks=[architecture_task, implementation_task, testing_task, reviewing_task],
    verbose=False
)
print("Devyan starting...")
print("Problem:", USER_PROBLEM)
print("=" * 60)
devyan_result = run_with_retry(devyan_crew)
if devyan_result:
    print("Devyan run complete.")


Devyan starting...
Problem: Build process_students(records: list[dict], min_score: float) -> dict that filters by min_score, sorts descending, returns {filtered, average, top_student}.


Devyan run complete.


In [ ]:
if devyan_result is None:
    print("No result — rate limit retries exhausted. Re-run the kickoff cell.")
else:
    labels = ["ARCHITECTURE", "IMPLEMENTATION", "TEST SUITE", "REVIEW & DOCS"]
    print("" + "=" * 60)
    print("DEVYAN — FINAL DELIVERABLES")
    print("=" * 60)
    for i, (label, output) in enumerate(zip(labels, devyan_result.tasks_output)):
        print(f"{chr(45)*60}")
        print(f"[{i+1}] {label}")
        print(f"{chr(45)*60}")
        print(str(output))


DEVYAN — FINAL DELIVERABLES
------------------------------------------------------------
[1] ARCHITECTURE
------------------------------------------------------------
**Function Signature**
```python
def process_students(records: list[dict], min_score: float) -> dict:
```

**Parameter Descriptions**

- `records`: A list of dictionaries containing student records.
- `min_score`: The minimum score to filter student records.

**Return Type**
```python
dict
```

**Return Value**
```python
{
    'filtered': list[dict],  # List of student records with scores >= min_score
    'average': float,  # Average score of filtered students
    'top_student': dict  # Student with the highest score
}
```

**Edge Cases**

1. **Empty Records**: `process_students([], 0.0)` returns `{'filtered': [], 'average': 0.0, 'top_student': None}`.
2. **No Students with Scores >= min_score**: `process_students([{'name': 'John', 'score': 0.0}], 1.0)` returns `{'filtered': [], 'average': 0.0, 'top_student': None}`.
3. *

---

## Summary

### Frameworks Comparison

| | AutoGen | CrewAI |
|---|---|---|
| **Abstraction** | Conversational agents | Role-based crew members |
| **Coordination** | Sequential `initiate_chat()` calls | Declarative `context=[task]` dependencies |
| **Tools** | `register_function()` | `@tool` decorator |
| **Best for** | Flexible multi-turn conversations | Structured pipelines with clear handoffs |
| **Execution** | One conversation at a time | `crew.kickoff()` runs all tasks |

### Design Patterns We Used

| Pattern | Where we used it |
|---|---|
| **Tool Use** | Researcher agent calling `search_internet` |
| **Planning** | Tasks defined upfront before execution |
| **Multi-Agent** | Researcher → Writer → Translator pipeline |
| **Specialization** | Each agent has a distinct role, goal, backstory |

### Key Takeaways

1. **Agents are just LLMs + a loop** — the loop adds planning, tool use, and error recovery.
2. **Specialization works** — a dedicated Researcher + Writer + Reviewer produces better output than a single all-in-one prompt.
3. **Tools are the bridge** to the real world — without tools, agents can only reason; with tools, they can act.
4. **The framework is scaffolding** — AutoGen and CrewAI both implement the same core patterns; choose based on your team's preference and the task structure.

### What's Next

In **Notebook 3** we'll tackle a critical question: *How do you know if your LLM is producing good outputs?* We'll use DeepEval and TruLens to systematically evaluate LLM and RAG system quality — and we'll look at the data, ethics, and security challenges behind these systems.